# 03 — Geometric Transforms (Ngày 9)

Notebook demo **visual so sánh before / after** cho từng phép biến đổi hình học trong `src/preprocessing/geometry.py`.

| Kỹ thuật | Hàm |
|---|---|
| Resize with padding (letterbox) | `resize_with_padding()` |
| Rotate | `rotate_image()` |
| Crop ROI | `crop_roi()` |
| Perspective correction | `correct_perspective()` |
| Affine transform | `apply_affine_transform()` |

In [ ]:
import sys
from pathlib import Path

# Thêm src/ vào path
ROOT = Path("__file__").resolve().parents[1]
sys.path.insert(0, str(ROOT / "src"))

import cv2
import matplotlib.pyplot as plt
import numpy as np

from preprocessing import (
    apply_affine_transform,
    correct_perspective,
    crop_roi,
    resize_with_padding,
    rotate_image,
)

print("Import OK ✓")

In [ ]:
def show_before_after(original, transformed, title_before="Before", title_after="After", figsize=(12, 5)):
    """Hiển thị ảnh trước và sau transform side-by-side."""
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    for ax, img, title in zip(axes, [original, transformed], [title_before, title_after]):
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(f"{title}\n{img.shape[1]}×{img.shape[0]} px", fontsize=12)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


def make_test_image(h: int = 480, w: int = 640) -> np.ndarray:
    """Tạo ảnh test với gradient + hình học để dễ quan sát transform."""
    img = np.zeros((h, w, 3), dtype=np.uint8)
    # Gradient nền
    for i in range(h):
        img[i, :, 0] = int(i / h * 180)   # Blue channel
    for j in range(w):
        img[:, j, 1] = int(j / w * 180)   # Green channel
    # Hình chữ nhật trắng ở giữa
    cv2.rectangle(img, (w//4, h//4), (3*w//4, 3*h//4), (200, 200, 200), 3)
    # Đường chéo đỏ
    cv2.line(img, (0, 0), (w, h), (0, 0, 220), 2)
    cv2.line(img, (w, 0), (0, h), (0, 0, 220), 2)
    # Text
    cv2.putText(img, "ORIGINAL", (w//2 - 80, h//2),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2)
    return img


img = make_test_image()
print(f"Test image shape: {img.shape}")

## 1. Resize with Padding (Letterbox)

Resize ảnh 640×480 về 640×640 **mà không méo tỷ lệ** bằng cách thêm padding đen.

In [ ]:
padded = resize_with_padding(img, target_size=(640, 640))
print(f"Input : {img.shape}  →  Output : {padded.shape}")
show_before_after(img, padded,
                  title_before=f"Original ({img.shape[1]}×{img.shape[0]})",
                  title_after=f"Letterbox 640×640")

In [ ]:
# So sánh nhiều target_size
targets = [(320, 320), (416, 416), (640, 640), (1280, 720)]
fig, axes = plt.subplots(1, len(targets), figsize=(16, 4))
for ax, ts in zip(axes, targets):
    result = resize_with_padding(img, target_size=ts)
    ax.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{ts[0]}×{ts[1]}", fontsize=10)
    ax.axis("off")
plt.suptitle("resize_with_padding — các target_size", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 2. Rotate Image

Xoay ảnh quanh tâm, giữ nguyên kích thước.

In [ ]:
angles = [15, 45, 90, 180]
fig, axes = plt.subplots(1, len(angles) + 1, figsize=(16, 4))
axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
axes[0].set_title("Original", fontsize=10)
axes[0].axis("off")

for ax, angle in zip(axes[1:], angles):
    rotated = rotate_image(img, angle=angle)
    ax.imshow(cv2.cvtColor(rotated, cv2.COLOR_BGR2RGB))
    ax.set_title(f"angle={angle}°", fontsize=10)
    ax.axis("off")

plt.suptitle("rotate_image — các góc xoay", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 3. Crop ROI

In [ ]:
# Crop vùng trung tâm
h, w = img.shape[:2]
roi = crop_roi(img, x=w//4, y=h//4, w=w//2, h=h//2)
show_before_after(img, roi,
                  title_before="Original — vùng ROI được đánh dấu",
                  title_after=f"Cropped ROI ({roi.shape[1]}×{roi.shape[0]})")

# Vẽ hộp ROI lên ảnh gốc để minh họa
img_marked = img.copy()
cv2.rectangle(img_marked, (w//4, h//4), (3*w//4, 3*h//4), (0, 255, 0), 3)
show_before_after(img_marked, roi,
                  title_before="Vùng ROI (khung xanh)",
                  title_after="ROI sau khi crop")

In [ ]:
# Demo ValueError khi vượt biên
try:
    bad_crop = crop_roi(img, x=w - 50, y=0, w=200, h=100)
except ValueError as e:
    print(f"✓ ValueError bắt đúng: {e}")

## 4. Perspective Correction

Chỉnh ảnh chụp nghiêng về dạng top-down (bird's eye view).

In [ ]:
# Mô phỏng ảnh chụp nghiêng bằng cách tạo hình thang
road_img = make_test_image(h=480, w=640)

# 4 điểm mô phỏng lane markings nhìn từ camera nghiêng
src_pts = np.array([
    [150, 100],   # top-left
    [490, 100],   # top-right
    [580, 380],   # bottom-right
    [ 60, 380],   # bottom-left
], dtype=np.float32)

# Vẽ quadrilateral lên ảnh gốc
road_marked = road_img.copy()
pts_int = src_pts.astype(np.int32)
cv2.polylines(road_marked, [pts_int], isClosed=True, color=(0, 255, 255), thickness=3)
for pt in pts_int:
    cv2.circle(road_marked, tuple(pt), 8, (0, 0, 255), -1)

warped = correct_perspective(road_img, src_pts, output_size=(400, 300))
show_before_after(road_marked, warped,
                  title_before="Original (4 điểm góc được đánh dấu)",
                  title_after=f"Perspective corrected ({warped.shape[1]}×{warped.shape[0]})")

## 5. Affine Transform

Biến đổi affine (shear, translate, scale) từ 3 cặp điểm tương ứng — giữ nguyên tính song song của các đường thẳng.

In [ ]:
h, w = img.shape[:2]

src_pts = np.float32([[0, 0], [w - 1, 0], [0, h - 1]])
# Shear nhẹ sang phải
dst_pts = np.float32([[w * 0.1, 0], [w - 1, 0], [0, h - 1]])

affined = apply_affine_transform(img, src_pts, dst_pts)
show_before_after(img, affined,
                  title_before="Original",
                  title_after="Affine Transform (shear)")

## 6. Pipeline Demo — kết hợp các transform

Mô phỏng pipeline tiền xử lý đầy đủ cho một frame video giao thông.

In [ ]:
import random

rng = np.random.default_rng(2024)
frame = rng.integers(0, 256, (720, 1280, 3), dtype=np.uint8)

# Giả lập bounding box phương tiện
vehicle_x, vehicle_y, vehicle_w, vehicle_h = 300, 200, 400, 250
cv2.rectangle(frame, (vehicle_x, vehicle_y),
              (vehicle_x + vehicle_w, vehicle_y + vehicle_h),
              (0, 255, 0), 3)

# Bước 1: Crop phương tiện
vehicle_crop = crop_roi(frame, vehicle_x, vehicle_y, vehicle_w, vehicle_h)

# Bước 2: Data augmentation — rotate ngẫu nhiên
angle = rng.integers(-15, 16)
augmented = rotate_image(vehicle_crop, angle=float(angle))

# Bước 3: Resize về YOLO input size
model_input = resize_with_padding(augmented, target_size=(640, 640))

steps = [
    (frame, f"1. Frame gốc (1280×720)"),
    (vehicle_crop, f"2. Crop ROI ({vehicle_w}×{vehicle_h})"),
    (augmented, f"3. Rotate {angle}° (augment)"),
    (model_input, "4. Letterbox 640×640 (YOLO input)"),
]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, (img_step, title) in zip(axes, steps):
    ax.imshow(cv2.cvtColor(img_step, cv2.COLOR_BGR2RGB))
    ax.set_title(title, fontsize=10)
    ax.axis("off")
plt.suptitle("Pipeline tiền xử lý đầy đủ", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print("Pipeline hoàn tất ✓")
print(f"Frame gốc  : {frame.shape}")
print(f"Model input: {model_input.shape}  dtype={model_input.dtype}")

## 7. Lưu ý kỹ thuật quan trọng

| ⚠ Lưu ý | Giải thích |
|---|---|
| **INTER_LINEAR vs INTER_AREA** | Dùng `INTER_LINEAR` khi phóng to, `INTER_AREA` khi thu nhỏ |
| **OpenCV vs NumPy coordinates** | OpenCV: `(x, y) = (col, row)` — NumPy: `[row, col]` |
| **crop_roi im lặng** | NumPy slicing `img[y:y+h, x:x+w]` không báo lỗi khi vượt biên — luôn validate trước |
| **borderMode khi rotate** | `BORDER_REPLICATE` giữ màu viền tốt hơn `BORDER_CONSTANT` (đen) |
| **warpAffine / warpPerspective** | Đối số `dsize` là `(width, height)` — ngược với NumPy `(height, width)` |